# 04 — Final Modeling Dataset

**Proje:** Credit Risk Scoring (Home Credit Default Risk)

Bu defterde **veri boru hattının son halkası**: tüm parça parquet'leri birleştirip modelleme defterleri için **tek bir dataset** üretiyoruz.

### Akış
1. **Base** = `application_train_bureau.parquet` (application_train + bureau, `307,511 × 335`)
2. Üzerine LEFT JOIN:
   - `previous_application_agg.parquet`
   - `pos_cash_agg.parquet`
   - `installments_payments_agg.parquet`
   - `credit_card_balance_agg.parquet`
3. **Domain feature'ları** ekle (`CREDIT_INCOME_PERCENT`, `ANNUITY_INCOME_PERCENT`, `CREDIT_TERM`, `DAYS_EMPLOYED_PERCENT`, `INCOME_PER_PERSON`).
4. `reduce_mem_usage` ile final downcast.
5. **Stratified train/valid split** (%80 / %20, `TARGET` üzerinde).
6. `data/processed/{train,valid}.parquet` olarak diske yaz.

### Tasarım kararları
- Train/valid split **fixed seed (42)** ile reproducible — sonraki tüm deney karşılaştırmaları aynı bölüm üzerinde olur.
- `SK_ID_CURR` final dataset'te tutuluyor (debug/explainability için), ama eğitimde feature olarak kullanılmamalı.
- `TARGET` her iki parquet'in son sütunu olarak korunur.

In [ ]:
import gc
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import reduce_mem_usage, missing_value_report, PROCESSED_DIR
from src.feature_engineering import add_application_features, merge_features

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

RANDOM_STATE = 42
TARGET = "TARGET"
ID_COL = "SK_ID_CURR"

print(f"Processed dir : {PROCESSED_DIR}")
print(f"Random state  : {RANDOM_STATE}")

## 1) Base'i yükle ve auxiliary parquet'leri birleştir

In [ ]:
base = pd.read_parquet(PROCESSED_DIR / "application_train_bureau.parquet")
print(f"base shape : {base.shape}")

aux_files = [
    "previous_application_agg.parquet",
    "pos_cash_agg.parquet",
    "installments_payments_agg.parquet",
    "credit_card_balance_agg.parquet",
]

dataset = base
for fname in aux_files:
    aux = pd.read_parquet(PROCESSED_DIR / fname)
    dataset = merge_features(dataset, aux, on=ID_COL)
    print(f"  + {fname:<42} -> {dataset.shape}")
    del aux
    gc.collect()

print(f"\nfinal merge shape: {dataset.shape}")

## 2) Domain feature'ları ekle ve belleği küçült

Klasik kredi riski oranları + `DAYS_EMPLOYED = 365243` anomalisinin temizlenmesi.

In [ ]:
dataset = add_application_features(dataset)
new_ratios = [
    "CREDIT_INCOME_PERCENT",
    "ANNUITY_INCOME_PERCENT",
    "CREDIT_TERM",
    "DAYS_EMPLOYED_PERCENT",
    "INCOME_PER_PERSON",
]
print("Domain ratio summary:")
print(dataset[new_ratios].describe().T[["mean", "std", "min", "max"]].round(3))

dataset = reduce_mem_usage(dataset, verbose=True)
print(f"\nfinal shape : {dataset.shape}")

## 3) Hızlı kontrol

- Toplam feature sayısı (TARGET ve ID hariç)
- En çok eksiği olan ilk 10 sütun
- TARGET dağılımı (split öncesi)

In [ ]:
feature_cols = [c for c in dataset.columns if c not in (TARGET, ID_COL)]
print(f"feature count    : {len(feature_cols)}")
print(f"TARGET balance   :")
print(dataset[TARGET].value_counts(normalize=True).round(4))

print("\nTop-10 missing columns:")
missing_value_report(dataset[feature_cols], top=10)

## 4) Stratified train/valid split

%80/%20, `TARGET` üzerinde stratify, `random_state=42` ile reproducible.

In [ ]:
train_df, valid_df = train_test_split(
    dataset,
    test_size=0.20,
    stratify=dataset[TARGET],
    random_state=RANDOM_STATE,
)

print(f"train shape : {train_df.shape}")
print(f"valid shape : {valid_df.shape}")
print(f"\nTARGET ratio (train) : {train_df[TARGET].mean():.4f}")
print(f"TARGET ratio (valid) : {valid_df[TARGET].mean():.4f}")

## 5) Diske kaydet

Modelleme defterleri (`05_Modeling_*.ipynb`) bu iki parquet'i doğrudan tüketecek.

In [ ]:
train_path = PROCESSED_DIR / "train.parquet"
valid_path = PROCESSED_DIR / "valid.parquet"

train_df.to_parquet(train_path, index=False)
valid_df.to_parquet(valid_path, index=False)

summary = pd.DataFrame(
    [
        {
            "file": train_path.name,
            "rows": len(train_df),
            "columns": train_df.shape[1],
            "size_mb": round(train_path.stat().st_size / 1024**2, 2),
            "target_rate": round(train_df[TARGET].mean(), 4),
        },
        {
            "file": valid_path.name,
            "rows": len(valid_df),
            "columns": valid_df.shape[1],
            "size_mb": round(valid_path.stat().st_size / 1024**2, 2),
            "target_rate": round(valid_df[TARGET].mean(), 4),
        },
    ]
).set_index("file")
summary